In [1]:
import natsort
import glob
from collections import  defaultdict
import tqdm 
import numpy as np 
import dask.array as da 
import dask 
import copy 

### 1. load filtered files

In [2]:
files = sorted(glob.glob("037_50/*"),key=natsort.natsort_key)

In [3]:
def split_files(files:list)->defaultdict:
    """
    split a list of files into a dictionary with {scan_no:{slice_no:filename.npz}}

    RETURNS
    dict
    """
    data = defaultdict(dict)
    for file in files:
        loc1 = int([x for x in file.split("_") if "pag" in x][0].replace("pag", ""))
        loc2 = int([x for x in file.split("_") if ".npz" in x][0].replace(".npz", ""))
        data[loc1][loc2] = file # np.load(file)["arr_0"][0]
    return data

In [4]:
splitted_files = split_files(files)

## 2. Load and further denoise the files 

avoids putting the files into RAM (as there is > 30GB per time step)

further denoises based on "slices" which are overly noisy (just converts them to 0s for now)

In [5]:
# mostly taken from gemini
def load_single_npz(path):
    with np.load(path, mmap_mode="r") as data:
        return data["arr_0"][0]


def get_virtual_volume(file_dict, volume_id=0):
    # 1. Get the slice paths for a specific volume
    slice_paths = file_dict[volume_id]
    num_slices = len(slice_paths)

    # 2. Get the shape of a single slice to define the 'virtual' metadata
    sample_shape = load_single_npz(slice_paths[0]).shape

    # 3. Create a list of "Delayed" objects (pointers to the files)
    lazy_slices = [
        da.from_delayed(
            dask.delayed(load_single_npz)(slice_paths[i]),
            shape=sample_shape,
            dtype="float32",
        )
        for i in range(num_slices)
    ]

    # 4. Stack them into a 3D Virtual Array
    # Still zero RAM used here!
    return da.stack(lazy_slices)


def get_denoised_stack(file_dict, volume_id=0, threshold=0.1):
    # 1. Get the dictionary of files for this specific volume
    # e.g., {0: "file0.npz", 1: "file1.npz", ...}
    slice_paths = file_dict[volume_id]
    num_slices = len(slice_paths)

    # 2. Peek at the first file to get the shape and dtype
    print("Peeking at file headers...")
    first_path = slice_paths[0]
    sample_data = load_single_npz(first_path)
    s_shape = sample_data.shape
    s_dtype = sample_data.dtype

    # 3. Create the Lazy Dask Stack
    lazy_slices = [
        da.from_delayed(
            dask.delayed(load_single_npz)(slice_paths[i]), shape=s_shape, dtype=s_dtype
        )
        for i in range(num_slices)
    ]
    virtual_stack = da.stack(lazy_slices)

    # 4. Pass 1: Calculate non-zero counts (Denoising Logic)
    # We do this in a loop with tqdm so you can see the progress
    print("Pass 1/2: Analyzing slice density for denoising...")
    counts = []
    for i in tqdm.tqdm(range(num_slices)):
        # We load only the metadata/nonzero count of one slice at a time
        path = slice_paths[i]
        with np.load(path, mmap_mode="r") as f:
            counts.append(np.count_nonzero(f["arr_0"][0]))

    counts = np.array(counts)
    mean_val = np.mean(counts)

    # Identify "Noisy" slices (those above threshold * mean)
    # 1.0 means 'keep', 0.0 means 'discard'
    keep_mask_raw = (counts <= threshold * mean_val).astype(s_dtype)

    # 5. Pass 2: Apply the mask lazily
    # Reshape mask from (Z,) to (Z, 1, 1) to allow broadcasting across X and Y
    dask_mask = da.from_array(keep_mask_raw[:, None, None], chunks=(1, 1, 1))

    # This is a virtual operation: no new large array is created in RAM
    return virtual_stack * dask_mask


In [11]:
volume = get_denoised_stack(splitted_files, volume_id=9, threshold=0.1)

small_vol = volume[::5, ::5, ::5]

Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


100%|██████████| 2016/2016 [00:20<00:00, 100.39it/s]


In [12]:
small_vol

dask.array<getitem, shape=(404, 404, 404), dtype=uint16, chunksize=(1, 404, 404), chunktype=numpy.ndarray>

## 3. Plot data 

In [13]:
import dask.array as da
import numpy as np
import pyvista as pv

def plot_lazy_volume(dask_arr, threshold_mask=None):
    """
    dask_arr: The lazy Dask array we created earlier.
    threshold_mask: Optional boolean array (True for keep, False for denoise/zero).
    """
    
    # 1. Apply slice-level denoising if you have a mask
    if threshold_mask is not None:
        # threshold_mask should be a 1D array of size (slices,)
        # We broadcast it to match the 3D dask_arr
        mask_3d = da.from_array(threshold_mask[:, None, None], chunks=dask_arr.chunks)
        dask_arr = dask_arr * mask_3d

    # 2. Find coordinates of non-zero points lazily
    # This creates a "graph" of how to find the points, doesn't do it yet.
    non_zero_indices = da.argwhere(dask_arr > 0)
    
    # 3. Compute ONLY the coordinates and their corresponding values
    # This is the moment data is read from disk.
    coords = non_zero_indices.compute()
    
    # Extract the values for the colors/scalars
    # We use the computed coords to index back into the dask array efficiently
    z, x, y = coords.T
    values = dask_arr.vindex[z, x, y].compute()

    # 4. PyVista Visualization
    points = pv.PolyData(coords)
    points["value"] = values


    plotter = pv.Plotter(window_size=(1000, 1000))
    plotter.add_mesh(
        points, 
        scalars="value", 
        render_points_as_spheres=True,
        point_size=2,
        cmap="viridis", # Changed to viridis to see the 'value' variation
        show_edges=False, 
        show_scalar_bar=True
    )
    plotter.add_bounding_box(color="black")
    #plotter.enable_eye_dome_lighting()
    plotter.show()

In [ ]:
plot_lazy_volume(small_vol)

/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/pyvista/core/utilities/points.py:79: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Widget(value='<iframe src="http://localhost:52128/index.html?ui=P_0x17396ac30_0&reconnect=auto" class="pyvista…

Loading Volume 1...
Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


100%|██████████| 2016/2016 [00:29<00:00, 68.27it/s]
/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/pyvista/core/utilities/points.py:79: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Loading Volume 1...
Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


100%|██████████| 2016/2016 [00:29<00:00, 68.60it/s]
/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/pyvista/core/utilities/points.py:79: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Loading Volume 0...
Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


100%|██████████| 2016/2016 [00:29<00:00, 69.21it/s]
/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/pyvista/core/utilities/points.py:79: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(
Exception raised
KeyError('4396ac884e3ad06dbc2918a477077c4b_2364f')
Traceback (most recent call last):
  File "/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/wslink/protocol.py", line 324, in onCompleteMessage
    results = func(*args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/trame_vtk/modules/vtk/protocols/local_rendering.py", line 33, in get_array
    self.context.get_cached_data_array(data_hash, binary)
  File "/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/trame_vtk/modules/vtk/serializers/synchronization_context.py", line 41, in get_cac

Loading Volume 0...
Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


 54%|█████▍    | 1084/2016 [00:15<00:14, 65.72it/s]

## 4. Testing  multiple 

In [14]:
import dask.array as da
import numpy as np
import pyvista as pv

class MultiVolumeVisualizer:
    def __init__(self, file_dict, threshold=0.1, downscale=2):
        self.file_dict = file_dict
        self.threshold = threshold
        self.downscale = downscale
        self.plotter = pv.Plotter(window_size=(1000, 1000))
        self.mesh_actor = None
        
        # Initialize with the first volume
        self.update_volume(0)
        
        # Add the interactive slider
        # (title, range, starting_val, callback_func)
        self.plotter.add_slider_widget(
            callback=self.update_volume,
            rng=[0, len(file_dict) - 1],
            value=0,
            title="Volume ID",
            event_type='always',
            fmt="%0.0f"
        )

    def get_lazy_data(self, volume_id):
        """Fetches the specific volume lazily using our Dask logic."""
        # This is where we call the 'get_denoised_stack' function we built earlier
        # Ensure it returns the downsampled version to keep the UI snappy
        vol = get_denoised_stack(self.file_dict, volume_id=int(volume_id), threshold=self.threshold)
        return vol[::self.downscale, ::self.downscale, ::self.downscale]

    def update_volume(self, volume_id):
        """Callback function for the slider."""
        print(f"Loading Volume {int(volume_id)}...")
        
        # 1. Get the lazy array
        lazy_arr = self.get_lazy_data(int(volume_id))
        
        # 2. Extract coordinates (The 'Compute' step)
        idx_task = da.argwhere(lazy_arr > 0)
        coords = idx_task.compute()
        
        if len(coords) == 0:
            print("Volume is empty after denoising!")
            return

        # 3. Get values safely
        z, x, y = coords.T
        values = lazy_arr.vindex[z, x, y].compute()
        
        # 4. Create Point Cloud
        point_cloud = pv.PolyData(coords)
        point_cloud["value"] = values
        
        # 5. Update Plotter
        if self.mesh_actor:
            self.plotter.remove_actor(self.mesh_actor)
            
        self.mesh_actor = self.plotter.add_mesh(
            point_cloud,
            scalars="value",
            render_points_as_spheres=True,
            point_size=3.0,
            cmap="viridis",
            name="vol_data" # Naming it helps PyVista overwrite efficiently
        )
        
        # Add depth perception
        self.plotter.enable_eye_dome_lighting()
        self.plotter.reset_camera()

    def show(self):
        self.plotter.show()

# --- To Run ---
# viz = MultiVolumeVisualizer(file_dict, threshold=0.1, downscale=2)
# viz.show()

In [15]:
import dask.array as da
import numpy as np
import pyvista as pv


class MultiVolumeVisualizer:
    def __init__(self, file_dict, threshold=0.1, downscale=2):
        self.file_dict = file_dict
        self.threshold = threshold
        self.downscale = downscale
        self.plotter = pv.Plotter(window_size=(1000, 1000))
        self.mesh_actor = None

        # Initialize with the first volume
        self.update_volume(0)

        # Add the interactive slider
        # (title, range, starting_val, callback_func)
        self.plotter.add_slider_widget(
            callback=self.update_volume,
            rng=[0, len(file_dict) - 1],
            value=0,
            title="Volume ID",
            # Use 'end' to only trigger when you let go of the slider
            # Use 'always' if your volume loads instantly and you want live sliding
            interaction_event="end",
            fmt="%0.0f",
        )

    def get_lazy_data(self, volume_id):
        """Fetches the specific volume lazily using our Dask logic."""
        # This is where we call the 'get_denoised_stack' function we built earlier
        # Ensure it returns the downsampled version to keep the UI snappy
        vol = get_denoised_stack(
            self.file_dict, volume_id=int(volume_id), threshold=self.threshold
        )
        return vol[:: self.downscale, :: self.downscale, :: self.downscale]

    def update_volume(self, volume_id):
        """Callback function for the slider."""
        print(f"Loading Volume {int(volume_id)}...")

        # 1. Get the lazy array
        lazy_arr = self.get_lazy_data(int(volume_id))

        # 2. Extract coordinates (The 'Compute' step)
        idx_task = da.argwhere(lazy_arr > 0)
        coords = idx_task.compute()

        if len(coords) == 0:
            print("Volume is empty after denoising!")
            return

        # 3. Get values safely
        z, x, y = coords.T
        values = lazy_arr.vindex[z, x, y].compute()

        # 4. Create Point Cloud
        point_cloud = pv.PolyData(coords)
        point_cloud["value"] = values

        # 5. Update Plotter
        if self.mesh_actor:
            self.plotter.remove_actor(self.mesh_actor)

        self.mesh_actor = self.plotter.add_mesh(
            point_cloud,
            scalars="value",
            render_points_as_spheres=True,
            point_size=3.0,
            cmap="viridis",
            name="vol_data",  # Naming it helps PyVista overwrite efficiently
        )

        # Add depth perception
        self.plotter.enable_eye_dome_lighting()
        self.plotter.reset_camera()

    def show(self):
        self.plotter.show()





In [16]:
# --- To Run ---
viz = MultiVolumeVisualizer(splitted_files, threshold=0.1, downscale=10)
viz.show()

Loading Volume 0...
Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


100%|██████████| 2016/2016 [00:29<00:00, 69.22it/s]
/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/pyvista/core/utilities/points.py:79: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Loading Volume 0...
Peeking at file headers...
Pass 1/2: Analyzing slice density for denoising...


100%|██████████| 2016/2016 [00:28<00:00, 69.66it/s]
/Users/badw/miniconda3/envs/py312/lib/python3.12/site-packages/pyvista/core/utilities/points.py:79: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Widget(value='<iframe src="http://localhost:52128/index.html?ui=P_0x173809940_1&reconnect=auto" class="pyvista…